In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('bank-full.csv', sep=';')

In [3]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [4]:
columns_to_use = ['age', 'job', 'marital', 'education',
                'balance', 'housing', 'contact', 'day',
                'month', 'duration', 'campaign', 'pdays',
                'previous', 'poutcome', 'y']

df = df[columns_to_use]

In [5]:
# checking for missing values

df.isnull().sum()

age          0
job          0
marital      0
education    0
balance      0
housing      0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [6]:
# Q1: mode of education

df.education.mode()

0    secondary
Name: education, dtype: object

In [8]:
# Q2: correlation matrix

df.corr(numeric_only=True)

,age,balance,day,duration,campaign,pdays,previous
age,1.000000,0.097783,-0.009120,-0.004648,0.004760,-0.023758,0.001288
balance,0.097783,1.000000,0.004503,0.021560,-0.014578,0.003435,0.016674
day,-0.009120,0.004503,1.000000,-0.030206,0.162490,-0.093044,-0.051710
duration,-0.004648,0.021560,-0.030206,1.000000,-0.084570,-0.001565,0.001203
campaign,0.004760,-0.014578,0.162490,-0.084570,1.000000,-0.088628,-0.032855
pdays,-0.023758,0.003435,-0.093044,-0.001565,-0.088628,1.000000,0.454820
previous,0.001288,0.016674,-0.051710,0.001203,-0.032855,0.454820,1.000000


In [9]:
# target encoding

df.y = (df.y == 'yes').astype(int)
df.head()

,age,job,marital,education,balance,housing,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,2143,yes,unknown,5,may,261,1,-1,0,unknown,0
1,44,technician,single,secondary,29,yes,unknown,5,may,151,1,-1,0,unknown,0
2,33,entrepreneur,married,secondary,2,yes,unknown,5,may,76,1,-1,0,unknown,0
3,47,blue-collar,married,unknown,1506,yes,unknown,5,may,92,1,-1,0,unknown,0
4,33,unknown,single,unknown,1,no,unknown,5,may,198,1,-1,0,unknown,0


In [10]:
# split the data

SEED = 42

from sklearn.model_selection import train_test_split

In [12]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=SEED)
df_train, df_val = train_test_split(df_full_train, test_size=0.2, random_state=SEED)

In [13]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [14]:
y_train = df_train.y.values
y_val = df_val.y.values
y_test = df_test.y.values

In [17]:
from sklearn.metrics import mutual_info_score

In [16]:
def calculate_mi(series):
    return mutual_info_score(series, df_train.y)

In [18]:
categorical_columns = ['job', 'marital', 'education', 'housing', 'contact', 'month', 'poutcome']

In [20]:
df_mi = df_train[categorical_columns].apply(calculate_mi)
df_mi = df_mi.sort_values(ascending=False).to_frame(name='MI')
df_mi

,MI
poutcome,0.029389
month,0.024972
contact,0.013437
housing,0.010465
job,0.007172
education,0.002777
marital,0.002019


In [21]:
df_train = df_train.drop('y', axis=1)
df_val = df_val.drop('y', axis=1)
df_test = df_test.drop('y', axis=1)

In [22]:
# Q4: logistic regression

from sklearn.linear_model import LogisticRegression

In [24]:
from sklearn.feature_extraction import DictVectorizer

In [25]:
# one hot encoding categorical features

dv = DictVectorizer(sparse=False)
train_dict = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

In [28]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [29]:
val_dict = df_val.to_dict(orient='records')
X_val = dv.fit_transform(val_dict)

In [30]:
y_pred = model.predict(X_val)

In [31]:
from sklearn.metrics import accuracy_score

In [35]:
original_score = accuracy_score(y_val, y_pred)

In [36]:
# Q5: 

features = df_train.columns.to_list()

scores = pd.DataFrame(columns=['eliminated_feature', 'accuracy', 'difference'])
for feature in features:
    subset = features.copy()
    subset.remove(feature)
    
    dv = DictVectorizer(sparse=False)
    train_dict = df_train[subset].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    model = LogisticRegression(solver='liblinear', max_iter=1000, C=1.0, random_state=SEED)
    model.fit(X_train, y_train)
    
    val_dict = df_val[subset].to_dict(orient='records')
    X_val = dv.transform(val_dict)
    
    y_pred = model.predict(X_val)
    score = accuracy_score(y_val, y_pred)
    
    scores.loc[len(scores)] = [feature, score, original_score - score]

In [37]:
scores

,eliminated_feature,accuracy,difference
0,age,0.901299,0.000000
1,job,0.901161,0.000138
2,marital,0.901991,-0.000691
3,education,0.901299,0.000000
4,balance,0.901161,0.000138
5,housing,0.903096,-0.001797
6,contact,0.901438,-0.000138
7,day,0.900608,0.000691
8,month,0.900470,0.000829
9,duration,0.889964,0.011335


In [38]:
scores[scores.index == scores.difference.abs().idxmin()]

,eliminated_feature,accuracy,difference
0,age,0.901299,0.0


In [41]:
# Q6

dv = DictVectorizer(sparse=False)
train_dict = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val.to_dict(orient='records')
X_val = dv.transform(val_dict)

scores = {}
for C in [0.01, 0.1, 1, 10, 100]:
    model = LogisticRegression(solver='liblinear', max_iter=1000, C=C, random_state=SEED)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    
    score = accuracy_score(y_val, y_pred)
    scores[C] = round(score, 3)
    print(f'C = {C}:\t Accuracy = {score}')

C = 0.01:	 Accuracy = 0.8989494055847387
C = 0.1:	 Accuracy = 0.9012994194083495
C = 1:	 Accuracy = 0.901022947193807
C = 10:	 Accuracy = 0.9012994194083495
C = 100:	 Accuracy = 0.9012994194083495


In [42]:
print(f'The smallest `C` is {max(scores, key=scores.get)}.')

The smallest `C` is 0.1.
